# 2. CSVLoader

The loader for **spreadsheet-style data** — CSV files (Comma-Separated Values), like an exported
Excel sheet or a database dump.

---

## 1. Simple Definition

> **Kid version:** Imagine a big table of stickers — each **row** is one sticker with details (name,
> color, price). `CSVLoader` takes that table and turns **each row into its own little box**
> (`Document`), so the computer can look at stickers one at a time.

**Professional definition:** `CSVLoader` reads a `.csv` file and, **by default, creates one
`Document` per row**. Each row's cells are turned into a `"column: value"` text block, and the row
number + file are stored in metadata.

```python
from langchain_community.document_loaders import CSVLoader

docs = CSVLoader("people.csv").load()
print(len(docs))                 # = number of data rows
print(docs[0].page_content)      # "name: Alice\nage: 30\ncity: Paris"
print(docs[0].metadata)          # {"source": "people.csv", "row": 0}
```

Example `people.csv`:

```
name,age,city
Alice,30,Paris
Bob,25,Tokyo
```

---

## 2. Why Does It Exist?

**The problem:** CSVs are tables. If you dumped the whole file into one blob of text, the LLM would
struggle to reason about individual records, and you couldn't filter/cite a specific row.

### Before LangChain

```python
import csv
rows = list(csv.DictReader(open("people.csv")))
# rows is a list of dicts. Now you must manually turn each into text,
# add metadata, track row numbers... every time.
```

### After LangChain

```python
docs = CSVLoader("people.csv").load()
# Each row → a Document with readable "col: value" text + {"row": n} metadata.
```

**One row = one document** is powerful for retrieval: you can embed each record separately, so a
search for "who lives in Tokyo?" pulls back exactly Bob's row, with metadata telling you it was row 1.

---

## 3. Real-Life Analogy

A **deck of index cards** 🗂️. A spreadsheet is a big table; `CSVLoader` copies **each row onto its
own index card**. Now you can shuffle, search, and pull out a single card (record) instead of
scanning the whole table every time.

Or an **address book**: each contact is its own entry (card), not one giant paragraph.

---

## 4. Where It Fits in LangChain Architecture

```
BaseLoader
    │
    ▼
CSVLoader          ← reads CSV → one Document PER ROW (by default)
```

- **`BaseLoader` → `CSVLoader`:** inherits `.load()`/`.lazy_load()`; implements "parse rows → many
  `Document`s."
- Key contrast with `TextLoader`: `TextLoader` = 1 file → 1 document; `CSVLoader` = 1 file → **many**
  documents (one per row).

---

## 5. Internal Working

```
  "people.csv"
        │
        ▼
  READ header row → ["name", "age", "city"]
        │
        ▼
  FOR EACH data row:
     Alice,30,Paris
        │
        ▼  format cells as "column: value"
     "name: Alice
      age: 30
      city: Paris"
        │
        ▼  wrap in a Document
     Document(page_content=<above>, metadata={"source": "people.csv", "row": 0})
        │
        ▼
  return [ Document(row0), Document(row1), ... ]
```

---

## 6. Attributes (constructor arguments)

### `file_path`

**Definition:** Path to the CSV file.

**Why it exists:** The source to read.

In [ ]:
from pprint import pprint

def pretty_print_doc(doc):
    print("=" * 80)
    print("📄 CONTENT")
    print("-" * 80)
    print(doc.page_content)

    print("\n🏷️ METADATA")
    print("-" * 80)
    pprint(doc.metadata)

    print("=" * 80)
    print()

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path = r"knowledge-source\organizations.csv")
loader

In [ ]:
docs = loader.load()
pretty_print_doc(docs[0])

### `source_column`

**Definition:** The name of a column whose value becomes each Document's `source` metadata (instead
of the file path).

**Why it exists:** For **citations** — you may want the source to be a URL or ID stored in a column,
not just the filename.

**When developers use it:** When rows have their own identity/link (e.g. an `article_url` column).

**Real-life use case:** Each index card citing its original web page rather than "the big table."


In [ ]:
# Here Industry column from csv will become the source metadata

from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path = r"knowledge-source\organizations.csv",
                   source_column="Industry")
docs = loader.load()
pretty_print_doc(docs[0])
pretty_print_doc(docs[1])

In [ ]:
# Here Number of employees column from csv will become the source metadata

from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path = r"knowledge-source\organizations.csv",
                   source_column="Number of employees")
docs = loader.load()
pretty_print_doc(docs[0])
pretty_print_doc(docs[1])

### `metadata_columns`

**Definition:** Columns to store in `metadata` instead of putting them in `page_content`.

**Why it exists:** Some columns (IDs, dates, categories) are better as **filterable metadata** than as
text the model reads.

**When developers use it:** When you want to filter searches by a field (e.g. only `category=news`).

**Real-life use case:** Writing the date/category on the *label* of the card, not in the body.

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path = r"knowledge-source\organizations.csv",
                   source_column="Industry",
                   metadata_columns=["Founded", "Country"])
docs = loader.load()
pretty_print_doc(docs[0])

### `csv_args`

**Definition:** A dict of options passed straight to Python's `csv.DictReader` — e.g. `delimiter`,
`quotechar`, `fieldnames`.

**Why it exists:** Real CSVs aren't always comma-separated. Semicolons, tabs (`\t`), and custom
quoting are common (especially European exports).

**When developers use it:** Non-standard separators or missing headers.

**Real-life use case:** Telling the reader "the columns here are separated by semicolons, not commas."

In [ ]:
loader = CSVLoader(file_path=r"knowledge-source\organizations.csv",
                   source_column="Industry",
                   metadata_columns=["Founded", "Country"],
                   csv_args={"delimiter": ","})

docs = loader.load()
pretty_print_doc(docs[0])

In [ ]:
# Example 1: Semicolon-separated CSV

loader = CSVLoader(file_path=r"knowledge-source\organizations.csv",
                   source_column="Industry",
                   metadata_columns=["Founded", "Country"],
                   csv_args={"delimiter": ";"})

docs = loader.load()
pretty_print_doc(docs[0])

In [ ]:
# Example 2: Tab-separated file (TSV)

loader = CSVLoader(
    file_path=r"knowledge-source\organizations.csv",
    source_column="Industry",
    metadata_columns=["Founded", "Country"],
    csv_args={"delimiter": "\t"}
)

docs = loader.load()
pretty_print_doc(docs[0])

In [ ]:
# Example 3: CSV without a header row

loader = CSVLoader(
    file_path=r"knowledge-source\organizations.csv",
    source_column="Industry",
    metadata_columns=["Founded", "Country"],
    csv_args={"fieldnames": ["Name", "Age", "City"]}
)

docs = loader.load()
pretty_print_doc(docs[0])

### `encoding`

**Definition:** File character encoding (e.g. `"utf-8"`).

**Why it exists:** Same as `TextLoader` — avoid decode errors / garbled characters.

In [10]:
loader = CSVLoader(file_path=r"knowledge-source\organizations.csv",
                   source_column="Industry",
                   metadata_columns=["Founded", "Country"],
                   encoding="utf-8")

docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
Index: 1
Organization Id: E84A904909dF528
Name: Liu-Hoover
Website: http://www.day-hartman.org/
Description: Ergonomic zero administration knowledge user
Industry: Online Publishing
Number of employees: 6852

🏷️ METADATA
--------------------------------------------------------------------------------
{'Country': 'Western Sahara',
 'Founded': '1980',
 'row': 0,
 'source': 'Online Publishing'}



### `content_columns`

**Definition:** Restrict `page_content` to only these columns.

**Why it exists:** You may only want a couple of columns embedded, ignoring the rest.

In [11]:
loader = CSVLoader(file_path=r"knowledge-source\organizations.csv",
                   source_column="Industry",
                   metadata_columns=["Founded", "Country"],
                   encoding="utf-8",
                   content_columns=["Description"])

docs = loader.load()
pretty_print_doc(docs[0])

📄 CONTENT
--------------------------------------------------------------------------------
Description: Ergonomic zero administration knowledge user

🏷️ METADATA
--------------------------------------------------------------------------------
{'Country': 'Western Sahara',
 'Founded': '1980',
 'row': 0,
 'source': 'Online Publishing'}

